## Parallel Perch + CNN Benchmark

Test whether splitting CPU cores between TF (Perch) and PyTorch (CNN) and running them in parallel threads saves wall time vs sequential.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Detect CPUs and split between TF and PyTorch
N_CPUS = os.cpu_count()
N_TF = max(1, N_CPUS // 2)
N_PT = max(1, N_CPUS - N_TF)
print(f"Total CPUs : {N_CPUS}", flush=True)
print(f"TF threads : {N_TF}  (Perch)", flush=True)
print(f"PT threads : {N_PT}  (CNN)", flush=True)

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # force CPU

_WHL = Path("/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel")
if _WHL.exists():
    whl = sorted(_WHL.glob("tensorflow*.whl"))
    if whl:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                str(whl[0]),
                "-q",
                "--no-deps",
                "--force-reinstall",
            ],
            check=True,
        )
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "keras",
                "tf_keras",
                "-q",
                "--no-deps",
            ],
            check=False,
        )
        print(f"Installed TF from {whl[0].name}", flush=True)

In [ ]:
# CRITICAL: set TF thread limits RIGHT AFTER import, before any graph execution.
# Once TF runs its first op the thread pool is locked — setting it later is a no-op.
import tensorflow as tf

tf.config.threading.set_intra_op_parallelism_threads(N_TF)
tf.config.threading.set_inter_op_parallelism_threads(2)
print(f"TF {tf.__version__}", flush=True)
print(
    f"TF intra-op threads : {tf.config.threading.get_intra_op_parallelism_threads()}",
    flush=True,
)
print(
    f"TF inter-op threads : {tf.config.threading.get_inter_op_parallelism_threads()}",
    flush=True,
)

In [ ]:
import gc
import importlib
import subprocess
import sys
import threading
import time
import warnings
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")


def _ensure(pkg, import_name=None):
    if importlib.util.find_spec(import_name or pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)


_ensure("timm")
_ensure("librosa")

import librosa
import timm

torch.set_num_threads(N_PT)
print(f"PyTorch threads : {torch.get_num_threads()}", flush=True)
print(f"timm {timm.__version__}  librosa {librosa.__version__}", flush=True)

In [ ]:
# Audio constants
SR = 32_000
WINDOW_SAMPLES = SR * 5  # 160,000
FILE_SAMPLES = SR * 60  # 1,920,000
N_WINDOWS = 12
BATCH_FILES = 16
CNN_CLIP = SR * 5
CNN_FMAX = 16_000
N_DRY_RUN = 20

COMP_BASE = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")

# Prefer real test soundscapes; fall back to train files for dry run
sc_dir = COMP_BASE / "test_soundscapes"
if not any(sc_dir.glob("*.ogg")):
    sc_dir = COMP_BASE / "train_soundscapes"
    print(f"No test soundscapes — dry-run on {N_DRY_RUN} train files", flush=True)

test_paths = sorted(sc_dir.glob("*.ogg"))[:N_DRY_RUN]
print(f"Benchmark files : {len(test_paths)}", flush=True)
print(f"First file      : {test_paths[0].name}", flush=True)

In [ ]:
# Load Perch SavedModel
print("Loading Perch...", flush=True)
t0 = time.perf_counter()
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn = birdclassifier.signatures["serving_default"]
print(f"Perch loaded in {time.perf_counter() - t0:.1f}s", flush=True)

# XLA warmup — compiles once so benchmarks don't pay compilation cost
print("XLA warmup (1 batch)...", flush=True)
t0 = time.perf_counter()
_dummy_batch = np.zeros((BATCH_FILES * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
_out = infer_fn(inputs=tf.convert_to_tensor(_dummy_batch))
_ = _out["label"].numpy()
del _dummy_batch, _out
gc.collect()
print(f"XLA warmup done in {time.perf_counter() - t0:.1f}s", flush=True)
print(
    f"Verify TF intra threads still: {tf.config.threading.get_intra_op_parallelism_threads()}",
    flush=True,
)

In [ ]:
import glob as _glob


# CNN model (must match soundscape-v7 training architecture)
class _GEMFreqPool(nn.Module):
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=2).pow(1.0 / p)


class _AttSEDHead(nn.Module):
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(feat_dim, feat_dim), nn.ReLU(True), nn.Dropout(dropout))
        self.att_conv = nn.Conv1d(feat_dim, num_classes, 1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, 1)

    def forward(self, x):
        x = self.fc(x.permute(0, 2, 1)).permute(0, 2, 1)
        att = F.softmax(torch.tanh(self.att_conv(x)), dim=-1)
        return torch.sigmoid((att * self.cls_conv(x)).sum(dim=-1))


class BirdModelBaseline(nn.Module):
    def __init__(self, backbone, n_classes, dropout=0.1):
        super().__init__()
        self.encoder = timm.create_model(backbone, pretrained=False, num_classes=0, global_pool="", in_chans=3)
        with torch.no_grad():
            feat_dim = self.encoder(torch.zeros(1, 3, 64, 128)).shape[1]
        self.gem_pool = _GEMFreqPool()
        self.head = _AttSEDHead(feat_dim, n_classes, dropout)

    def forward(self, x):
        return self.head(self.gem_pool(self.encoder(x)))


# Load first checkpoint
ckpt_paths = sorted(Path(p) for p in _glob.glob("/kaggle/input/**/*.pt", recursive=True))
print(f"Checkpoints found: {len(ckpt_paths)}", flush=True)
if not ckpt_paths:
    raise RuntimeError("No .pt files found — attach aldisued/birdclef2026-soundscape-v7 dataset")

ckpt0 = torch.load(ckpt_paths[0], map_location="cpu", weights_only=False)
_backbone = ckpt0.get("backbone", "tf_efficientnet_b0.ns_jft_in1k")
_n_mels = ckpt0.get("n_mels", 224)
_n_fft = ckpt0.get("n_fft", 2048)
_hop = ckpt0.get("hop_length", 512)
_fmin = ckpt0.get("fmin", 0)
_htk = ckpt0.get("htk", True)
_n_sp = ckpt0["model"]["head.cls_conv.bias"].shape[0]

cnn_model = BirdModelBaseline(backbone=_backbone, n_classes=_n_sp)
cnn_model.load_state_dict(ckpt0["model"])
cnn_model.eval()
del ckpt0
gc.collect()
print(f"CNN: {_backbone}  n_species={_n_sp}", flush=True)
print(f"Config: n_mels={_n_mels} htk={_htk} fmin={_fmin} hop={_hop}", flush=True)

In [ ]:
# Benchmark helpers


def bench_perch(paths, tag="Perch"):
    """Sequential Perch inference over all paths. Returns elapsed seconds."""
    paths = list(paths)
    batches = [paths[s : s + BATCH_FILES] for s in range(0, len(paths), BATCH_FILES)]
    t_start = time.perf_counter()
    for bi, batch in enumerate(batches):
        windows = []
        for p in batch:
            y, _ = sf.read(p, dtype="float32", always_2d=False)
            if y.ndim == 2:
                y = y.mean(axis=1)
            if len(y) < FILE_SAMPLES:
                y = np.pad(y, (0, FILE_SAMPLES - len(y)))
            y = y[:FILE_SAMPLES]
            for i in range(N_WINDOWS):
                windows.append(y[i * WINDOW_SAMPLES : (i + 1) * WINDOW_SAMPLES])
        x = np.stack(windows).astype(np.float32)
        out = infer_fn(inputs=tf.convert_to_tensor(x))
        _ = out["label"].numpy()
        del x, out, windows
        gc.collect()
        print(
            f"  [{tag}] batch {bi + 1}/{len(batches)}: {time.perf_counter() - t_start:.1f}s",
            flush=True,
        )
    return time.perf_counter() - t_start


@torch.no_grad()
def bench_cnn(paths, tag="CNN"):
    """Sequential CNN inference over all paths. Returns elapsed seconds."""
    paths = list(paths)
    t_start = time.perf_counter()
    for i, p in enumerate(paths):
        y, _ = librosa.load(p, sr=SR, mono=True)
        n_slots = len(y) // CNN_CLIP
        if n_slots == 0:
            continue
        chunks = []
        for s in range(n_slots):
            chunk = y[s * CNN_CLIP : (s + 1) * CNN_CLIP]
            mel = librosa.feature.melspectrogram(
                y=chunk,
                sr=SR,
                n_mels=_n_mels,
                hop_length=_hop,
                n_fft=_n_fft,
                fmin=_fmin,
                fmax=CNN_FMAX,
                htk=_htk,
            )
            spec = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
            mn, mx = spec.min(), spec.max()
            spec = (spec - mn) / (mx - mn + 1e-6)
            chunks.append(np.stack([spec, spec, spec]))
        batch = torch.from_numpy(np.stack(chunks))
        _ = cnn_model(batch).numpy()
        del batch, chunks
        elapsed = time.perf_counter() - t_start
        eta = elapsed / (i + 1) * (len(paths) - i - 1)
        print(
            f"  [{tag}] [{i + 1}/{len(paths)}]: {elapsed:.1f}s | ETA {eta:.0f}s",
            flush=True,
        )
    return time.perf_counter() - t_start

In [ ]:
print("=" * 60, flush=True)
print(f"BENCHMARK 1: Sequential Perch  (TF threads={N_TF})", flush=True)
print("=" * 60, flush=True)
t_perch_seq = bench_perch(test_paths)
n = len(test_paths)
print(
    f"\nPerch sequential : {t_perch_seq:.1f}s | {t_perch_seq / n:.2f}s/file | extrap 600 files: {t_perch_seq / n * 600 / 60:.1f} min",
    flush=True,
)

print(flush=True)
print("=" * 60, flush=True)
print(f"BENCHMARK 2: Sequential CNN  (PyTorch threads={N_PT})", flush=True)
print("=" * 60, flush=True)
t_cnn_seq = bench_cnn(test_paths)
print(
    f"\nCNN sequential   : {t_cnn_seq:.1f}s | {t_cnn_seq / n:.2f}s/file | extrap 600 files: {t_cnn_seq / n * 600 / 60:.1f} min",
    flush=True,
)

print(flush=True)
print(
    f"Sequential total : {t_perch_seq + t_cnn_seq:.1f}s = {(t_perch_seq + t_cnn_seq) / 60:.1f} min",
    flush=True,
)
print(f"Extrap 600 files : {(t_perch_seq + t_cnn_seq) / n * 600 / 60:.1f} min", flush=True)

In [ ]:
print("=" * 60, flush=True)
print(f"BENCHMARK 3: Parallel  (TF={N_TF} threads / PyTorch={N_PT} threads)", flush=True)
print("=" * 60, flush=True)

_par = {}


def _perch_worker():
    t = bench_perch(test_paths, tag="Perch-par")
    _par["perch"] = t
    print(f"[THREAD] Perch done: {t:.1f}s", flush=True)


def _cnn_worker():
    t = bench_cnn(test_paths, tag="CNN-par")
    _par["cnn"] = t
    print(f"[THREAD] CNN done: {t:.1f}s", flush=True)


t_wall_start = time.perf_counter()
th_p = threading.Thread(target=_perch_worker)
th_c = threading.Thread(target=_cnn_worker)
th_p.start()
th_c.start()
th_p.join()
th_c.join()
t_wall = time.perf_counter() - t_wall_start
print(f"\nParallel wall time : {t_wall:.1f}s = {t_wall / 60:.1f} min", flush=True)

In [ ]:
n = len(test_paths)
t_seq_total = t_perch_seq + t_cnn_seq
t_ideal = max(_par.get("perch", 0), _par.get("cnn", 0))

print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"CPUs: {N_CPUS}  |  TF threads: {N_TF}  |  PyTorch threads: {N_PT}")
print(f"Files benchmarked: {n}")
print()
print(f"{'Mode':<30} {'Total':>8}  {'s/file':>8}  {'extrap 600':>12}")
print("-" * 62)
print(f"{'Sequential Perch':<30} {t_perch_seq:>7.1f}s  {t_perch_seq / n:>7.2f}s  {t_perch_seq / n * 600 / 60:>11.1f}m")
print(f"{'Sequential CNN':<30} {t_cnn_seq:>7.1f}s  {t_cnn_seq / n:>7.2f}s  {t_cnn_seq / n * 600 / 60:>11.1f}m")
print(f"{'Sequential TOTAL':<30} {t_seq_total:>7.1f}s  {t_seq_total / n:>7.2f}s  {t_seq_total / n * 600 / 60:>11.1f}m")
print("-" * 62)
print(f"{'Parallel Perch thread':<30} {_par.get('perch', 0):>7.1f}s")
print(f"{'Parallel CNN thread':<30} {_par.get('cnn', 0):>7.1f}s")
print(f"{'Parallel WALL TIME':<30} {t_wall:>7.1f}s  {t_wall / n:>7.2f}s  {t_wall / n * 600 / 60:>11.1f}m")
print("-" * 62)
print(f"Speedup vs sequential : {t_seq_total / t_wall:.2f}x")
print(f"Ideal speedup (theory): {t_seq_total / t_ideal:.2f}x")
print()
overhead = t_wall - t_ideal
print(f"Parallel overhead (wall - max_thread): {overhead:.1f}s = {overhead / t_ideal * 100:.0f}%")
print()
if t_wall / n * 600 / 60 < 87:
    print(">>> VERDICT: FITS in 90 min — blend is viable!")
else:
    print(f">>> VERDICT: {t_wall / n * 600 / 60:.0f} min extrap — too slow for 90-min limit.")